# Day 14 — Code quality with Ruff and mypy
Objectives:
- Use Ruff to find correctness, import, and style problems.
- Use Ruff's formatter to keep layout consistent.
- Use mypy to check type contracts before runtime.
- Distinguish formatting, linting, type checking, and tests.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-14`. Read
`python/ds-60day/companion-guides/day14_code_quality_tooling.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Code-quality tools answer different questions. A formatter applies one
consistent layout. A linter detects selected suspicious or inconsistent
patterns. A static type checker compares annotated contracts without
executing the program. Tests still check runtime behavior; no one tool
replaces the others.

Treat tool output as a precise diagnostic: file, line, rule, and
message. Understand a warning before suppressing it, make the smallest
change, then rerun the narrow command. Configuration belongs in
`pyproject.toml` so developers and continuous integration use the same
rules.

### Vocabulary

- **formatter:** a tool that rewrites source layout consistently.
- **linter:** a static checker for selected code patterns and style rules.
- **type checker:** a tool that compares annotations and operations without running code.
- **diagnostic:** a tool report tied to a location and rule.
- **configuration:** shared settings controlling tool behavior.
- **suppression:** an explicit request to ignore one diagnostic, ideally with rationale.

## Syntax anatomy

`python -m ruff check path` chooses the repository interpreter, runs the
Ruff module, performs checks, and scopes them to `path`. `ruff format
--check` reports formatting drift without rewriting. A function
annotation such as `def total(values: list[float]) -> float:` gives
mypy a contract to compare with callers and return statements.

### Worked example 1 — Inspect annotations as documentation

Hints are stored but do not enforce runtime arguments by themselves. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
from typing import get_type_hints

def total(values: list[float]) -> float:
    return sum(values)

(get_type_hints(total), total([1.5, 2.0]))

**Expected observation:** `({'values': list[float], 'return': float}, 3.5)` (representation may vary slightly). A type checker uses the hints before execution.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Refactor a long expression into named facts

Formatting cannot choose meaningful names; design still belongs to the author. Predict first; then run the next cell.

In [ ]:
prices = [12.0, 8.0]
subtotal = sum(prices)
discount_rate = 0.10
discounted_total = subtotal * (1 - discount_rate)
round(discounted_total, 2)

**Expected observation:** `18.0`. A formatter controls whitespace; the names make the calculation explainable.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Read the diagnostic rule and location before using an automatic fix.
2. Run formatter, linter, type checker, and tests separately so the failing contract is clear.
3. Use the repository interpreter to avoid invoking a globally installed tool with different versions.
4. If suppression is necessary, scope it narrowly and explain why the code is safe.

**Alternative to compare:** Editor-on-save feedback is fast, while command-line checks are reproducible; continuous integration should run the same checked-in configuration.

**Boundary to test:** Generated files, notebook code, untyped third-party libraries, version drift, and broad suppressions can create misleading results.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

## Sample `pyproject.toml`
```toml
[tool.ruff]
target-version = 'py311'
line-length = 100

[tool.ruff.lint]
select = ['E', 'F', 'I', 'UP', 'B']

[tool.mypy]
python_version = '3.11'
strict = true
```
Run from the project root:
```text
python -m ruff check .
python -m ruff format --check .
python -m mypy .
```
A formatter changes layout; a linter finds suspicious code; a type checker verifies annotated contracts. None replaces tests.

## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Run Ruff formatting and lint checks on a small intentionally messy Python file. **Evidence:** record at least one formatter change and one lint diagnostic with its rule code. **Constraints:** understand each automatic fix, do not run broad fixes over unrelated repository files, and delete only the scratch file you created.
   **Verify:** both `ruff format --check` and `ruff check` pass afterward.

2. Add useful type hints to a function that accepts records and returns a numeric total, then run mypy on the file. **Introduce:** one wrong caller argument and one wrong return in scratch code so you can read both diagnostics.
   **Expected behavior:** mypy rejects both; after repair it reports success while runtime tests still pass.
   **Verify:** Save the two expected mypy diagnostics, repair both defects, and show mypy plus the runtime test command pass.

### Additional mastery practice

Use formatter, linter, type checker, and tests as complementary evidence. Read and fix one diagnostic at a time, then review behavior.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Given an unused import, inconsistent spacing, a possible `None` value, and a wrong result, predict which quality tool can detect each.
   **Progressive hint:** No single tool proves all four properties.
   **Verify:** Create a four-row matrix mapping each defect to formatter, linter, type checker, or test; run the tools and record which predictions were confirmed.
4. **Tracing:** Trace type narrowing for `str | None` through an explicit `is None` branch and state the type in each path.
   **Progressive hint:** A type checker follows control-flow evidence.
   **Verify:** Capture `reveal_type` or equivalent type-checker output in both branches; assert the non-None branch reports `str` and the absence branch reports an error for a string-method call.
5. **Implementation:** Refactor an untyped file loader into a typed function using `Path`, a context manager, UTF-8, and a validated return shape.
   **Progressive hint:** Boundary validation can replace an unhelpfully broad `Any`.
   **Verify:** Type-check the loader, then test valid UTF-8 input plus missing column, invalid field type, and missing path boundaries.
6. **Debugging:** Replace an unexplained `# type: ignore` with a real narrowing or a narrow error-code ignore plus justification.
   **Progressive hint:** Do not suppress diagnostics before understanding the contract.
   **Verify:** Show the original diagnostic, replace suppression with narrowing where possible, and confirm any remaining ignore names one error code and rationale.
7. **Edge case and explanation:** Design a local/CI gate order and explain why formatter success should not prevent tests from running during diagnosis.
   **Progressive hint:** Fast static checks give feedback, but each signal remains independent.
   **Verify:** Run each gate independently on a deliberately broken scratch file and record its signal; confirm one early failure does not erase later diagnostic evidence.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Run Ruff formatting and lint checks on a small intentionally messy Python file. **Evidence:** record at least one formatter change and one lint diagnostic with its rule code. **Constraints:** understand each automatic fix, do not run broad fixes over unrelated repository files, and delete only the scratch file you created. **Verify:** both `ruff format --check` and `ruff check` pass afterward.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Run Ruff formatting and lint checks on a small intentionally messy Python file. record at least one formatter change and one lint diagnostic with its rule code. understand each...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add useful type hints to a function that accepts records and returns a numeric total, then run mypy on the file. **Introduce:** one wrong caller argument and one wrong return in scratch code so you can read both diagnostics. **Expected behavior:** mypy rejects both; after repair it reports success while runtime tests still pass. **Verify:** Save the two expected mypy diagnostics, repair both defects, and show mypy plus the runtime test command pass.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add useful type hints to a function that accepts records and returns a numeric total, then run mypy on the file. one wrong caller argument and one wrong return in scratch code s...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Given an unused import, inconsistent spacing, a possible `None` value, and a wrong result, predict which quality tool can detect each. **Progressive hint:** No single tool proves all four properties. **Verify:** Create a four-row matrix mapping each defect to formatter, linter, type checker, or test; run the tools and record which predictions were confirmed.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Given an unused import, inconsistent spacing, a possible `None` value, and a wrong result, predict which quality tool can detect each. No single tool proves all four properties....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace type narrowing for `str | None` through an explicit `is None` branch and state the type in each path. **Progressive hint:** A type checker follows control-flow evidence. **Verify:** Capture `reveal_type` or equivalent type-checker output in both branches; assert the non-None branch reports `str` and the absence branch reports an error for a string-method call.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace type narrowing for `str | None` through an explicit `is None` branch and state the type in each path. A type checker follows control-flow evidence. Capture `reveal_type` o...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Refactor an untyped file loader into a typed function using `Path`, a context manager, UTF-8, and a validated return shape. **Progressive hint:** Boundary validation can replace an unhelpfully broad `Any`. **Verify:** Type-check the loader, then test valid UTF-8 input plus missing column, invalid field type, and missing path boundaries.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Refactor an untyped file loader into a typed function using `Path`, a context manager, UTF-8, and a validated return shape. Boundary validation can replace an unhelpfully broad...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Replace an unexplained `# type: ignore` with a real narrowing or a narrow error-code ignore plus justification. **Progressive hint:** Do not suppress diagnostics before understanding the contract. **Verify:** Show the original diagnostic, replace suppression with narrowing where possible, and confirm any remaining ignore names one error code and rationale.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Replace an unexplained `# type: ignore` with a real narrowing or a narrow error-code ignore plus justification. Do not suppress diagnostics before understanding the contract. Sh...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Design a local/CI gate order and explain why formatter success should not prevent tests from running during diagnosis. **Progressive hint:** Fast static checks give feedback, but each signal remains independent. **Verify:** Run each gate independently on a deliberately broken scratch file and record its signal; confirm one early failure does not erase later diagnostic evidence.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Design a local/CI gate order and explain why formatter success should not prevent tests from running during diagnosis. Fast static checks give feedback, but each signal remains...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
